# NB01 — ENIGMA Genome Depot Survey: Mycothiol-Dependent Malonylpyruvate Isomerase

Pangenomic enrichment analysis (`mycothiol_detox_module/NB04`) established that K16163 (mai / malonylpyruvate isomerase) is the most phylogenetically robust Actinobacteria-specific enrichment signal remaining after one-per-genus subsampling. The gene is typically syntenic with mshC (K03975) within the mycothiol biosynthesis locus.

The ENIGMA Genome Depot contains genomes of isolates from metal-contaminated Oak Ridge sites available for lab experiments. This notebook identifies which isolates carry MAI and assesses mycothiol pathway completeness.

**Target KOs**:
| KO | Gene | Function |
|---|---|---|
| K01687 | mshA | MshA glycosyltransferase |
| K09861 | mshB | MshB deacetylase |
| K03975 | mshC | cysteine ligase |
| K16163 | mai  | malonylpyruvate isomerase |

In [1]:
import sys, os, re, warnings, requests, json, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import fisher_exact
warnings.filterwarnings("ignore")

spark = get_spark_session()
spark.clearProgressHandlers()
import json as _json
class _SparkSafeEncoder(_json.JSONEncoder):
    def default(self, obj):
        try:
            return super().default(obj)
        except TypeError:
            return str(obj)
_json.JSONEncoder = _SparkSafeEncoder
_json._default_encoder = _SparkSafeEncoder()

from pyspark.sql import functions as F

NOTEBOOK_DIR = Path().resolve()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data"
FIG_DIR      = PROJECT_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

def is_valid_parquet(p):
    if not p.exists() or p.stat().st_size < 512:
        return False
    try:
        import pyarrow.parquet as pq; pq.read_schema(str(p)); return True
    except Exception:
        return False

print("Setup complete.")

Setup complete.


## Section 1 — All ENIGMA genomes with taxonomy

In [2]:
# Single comprehensive query: all ENIGMA genomes + mycothiol pathway KO flags
# KO table is in enigma_genome_depot_enigma.browser_kegg_ortholog / browser_protein_kegg_orthologs
enigma_path = DATA_DIR / "enigma_all_genomes.parquet"

if is_valid_parquet(enigma_path):
    enigma_df = pd.read_parquet(enigma_path)
    print(f"[cache] {len(enigma_df)} ENIGMA genomes")
else:
    enigma_df = spark.sql('''
        SELECT
            g.id AS genome_id,
            s.full_name AS organism_name,
            MAX(CASE WHEN ko.kegg_id = 'K01687' THEN 1 ELSE 0 END) AS has_mshA,
            MAX(CASE WHEN ko.kegg_id = 'K09861' THEN 1 ELSE 0 END) AS has_mshB,
            MAX(CASE WHEN ko.kegg_id = 'K03975' THEN 1 ELSE 0 END) AS has_mshC,
            MAX(CASE WHEN ko.kegg_id = 'K16163' THEN 1 ELSE 0 END) AS has_mai
        FROM enigma_genome_depot_enigma.browser_genome g
        JOIN enigma_genome_depot_enigma.browser_strain s ON s.id = g.strain_id
        LEFT JOIN enigma_genome_depot_enigma.browser_gene ge ON ge.genome_id = g.id
        LEFT JOIN enigma_genome_depot_enigma.browser_protein_kegg_orthologs pko
            ON pko.protein_id = ge.protein_id
        LEFT JOIN enigma_genome_depot_enigma.browser_kegg_ortholog ko
            ON ko.id = pko.kegg_ortholog_id
            AND ko.kegg_id IN ('K01687', 'K09861', 'K03975', 'K16163')
        GROUP BY g.id, s.full_name
    ''').toPandas()
    enigma_df.to_parquet(enigma_path, index=False)
    print(f"Queried {len(enigma_df)} ENIGMA genomes")

print(f"MAI-positive: {enigma_df['has_mai'].sum()}")
print(f"mshC-positive: {enigma_df['has_mshC'].sum()}")

Queried 2925 ENIGMA genomes
MAI-positive: 163
mshC-positive: 2189


## Section 2 — Search for pathway KOs

### 2a. KO-based search (primary)

In [3]:
# KO hit summary from the comprehensive query
ko_hits = enigma_df[enigma_df[['has_mshA','has_mshB','has_mshC','has_mai']].max(axis=1) > 0]
print("Count per KO:")
for col, ko in [('has_mshA','K01687'),('has_mshB','K09861'),('has_mshC','K03975'),('has_mai','K16163')]:
    print(f"  {ko}: {enigma_df[col].sum()} genomes")

Count per KO:
  K01687: 2829 genomes
  K09861: 2099 genomes
  K03975: 2189 genomes
  K16163: 163 genomes


### 2b. Product name rescue

In [4]:
# Product-name rescue not available via browser tables — KO-based search is primary
print("KO-based search is the primary detection method via browser_kegg_ortholog.")

KO-based search is the primary detection method via browser_kegg_ortholog.


### 2c. Merge search strategies

In [5]:
merged_genome_ids = enigma_df[enigma_df['has_mai'] == 1]['genome_id'].tolist()
print(f"KO-based (MAI): {enigma_df['has_mai'].sum()} genomes")
print(f"Any pathway KO: {len(ko_hits)} genomes")
print(f"MAI-positive (for downstream): {len(merged_genome_ids)} genomes")

KO-based (MAI): 163 genomes
Any pathway KO: 2857 genomes
MAI-positive (for downstream): 163 genomes


## Section 3 — Pathway completeness

In [6]:
# Pathway completeness from the comprehensive query
enigma_df['pathway_completeness'] = (
    enigma_df[['has_mshA', 'has_mshB', 'has_mshC', 'has_mai']].sum(axis=1) / 4.0
)
pathway_pivot = enigma_df.rename(columns={
    'has_mshA': 'mshA', 'has_mshB': 'mshB', 'has_mshC': 'mshC', 'has_mai': 'mai'
})

has_any = pathway_pivot[pathway_pivot['pathway_completeness'] > 0]
print(f"Genomes with any pathway KO: {len(has_any)}")
print(f"\nPathway completeness distribution:")
print(has_any['pathway_completeness'].value_counts().sort_index(ascending=False))

Genomes with any pathway KO: 2857

Pathway completeness distribution:
pathway_completeness
1.00     134
0.75    1776
0.50     469
0.25     478
Name: count, dtype: int64


In [7]:
mai_positive = pathway_pivot[pathway_pivot['mai'] == 1].copy()
print(f"MAI-positive genomes: {len(mai_positive)}")

if len(mai_positive) > 0:
    display_cols = ['organism_name', 'mshA', 'mshB', 'mshC', 'mai', 'pathway_completeness']
    print("\nMAI-positive genomes:")
    print(mai_positive[display_cols].to_string())
    mai_positive.to_csv(DATA_DIR / "enigma_mai_hits.csv", index=False)
    print(f"\nSaved to {DATA_DIR / 'enigma_mai_hits.csv'}")
else:
    mai_positive = pd.DataFrame()
    print("No MAI-positive genomes found.")

pathway_pivot.to_csv(DATA_DIR / "enigma_pathway_completeness.csv", index=False)
print(f"Saved full completeness table to {DATA_DIR / 'enigma_pathway_completeness.csv'}")

MAI-positive genomes: 163

MAI-positive genomes:
                                                                                         organism_name  mshA  mshB  mshC  mai  pathway_completeness
1                                                                 Paenarthrobacter aurescens FW305-123     1     1     1    1                  1.00
3                                                                Environmental isolate FW306-2-2C-D06B     1     1     1    1                  1.00
41                                                            Environmental isolate SSO-M6-C2-PAR-NZ99     1     0     0    1                  0.50
79                                                                 Ensifer adhaerens EB106-05-01-XG146     1     0     0    1                  0.50
114                                                                      Environmental isolate MPR-R1A     1     1     1    1                  1.00
120                                                            

In [ ]:
# De-duplication, taxonomy breakdown, and complete-pathway stats
# Reads from cached enigma_all_genomes.parquet — no Spark required
import pandas as pd
from scipy.stats import fisher_exact
from pathlib import Path as _Path

try:
    _data_dir = DATA_DIR  # already set by setup_cell when run in sequence
except NameError:
    _data_dir = _Path.cwd().parent / "data"

enigma_df_all = pd.read_parquet(_data_dir / "enigma_all_genomes.parquet")

# ── 1. Overall counts ────────────────────────────────────────────────────────
n_genomes    = len(enigma_df_all)
n_unique_org = enigma_df_all["organism_name"].nunique()
mai_all      = enigma_df_all[enigma_df_all["has_mai"] == 1]
n_mai_unique = mai_all["organism_name"].nunique()
print(f"Total genome records: {n_genomes:,}")
print(f"Unique organisms (de-duplicated on organism_name): {n_unique_org:,}")
print(f"MAI-positive genome records: {len(mai_all):,}")
print(f"MAI-positive unique strains: {n_mai_unique:,}")

# ── 2. Complete 4-gene pathway de-duplication ────────────────────────────────
complete = enigma_df_all[
    (enigma_df_all["has_mshA"] == 1) & (enigma_df_all["has_mshB"] == 1) &
    (enigma_df_all["has_mshC"] == 1) & (enigma_df_all["has_mai"] == 1)
]
n_complete_records = len(complete)
n_complete_unique  = complete["organism_name"].nunique()
print(f"\nComplete 4-gene pathway records: {n_complete_records}")
print(f"Complete 4-gene pathway unique strains: {n_complete_unique}")

# Per-genus breakdown for complete-pathway
env_mask_c = complete["organism_name"].str.startswith("Environmental isolate", na=True)
complete_env      = complete[env_mask_c]
complete_resolved = complete[~env_mask_c].copy()
complete_resolved["genus"] = complete_resolved["organism_name"].str.split().str[0]
n_complete_env_unique = complete_env["organism_name"].nunique()
print(f"\nComplete-pathway genus breakdown:")
print(f"  Environmental (unresolved): {n_complete_env_unique} unique strains")
cgenus = complete_resolved.groupby("genus")["organism_name"].nunique().sort_values(ascending=False)
for g, n in cgenus.items():
    print(f"  {g:<20} {n:>3} unique strains")

# ── 3. Taxonomy classification (resolved genus-level) ────────────────────────
env_mask = enigma_df_all["organism_name"].str.startswith("Environmental isolate", na=True)
resolved  = enigma_df_all[~env_mask].copy()
resolved["genus"] = resolved["organism_name"].str.split().str[0]

ACTINO = {
    "Streptomyces", "Arthrobacter", "Paenarthrobacter", "Mycobacterium",
    "Nocardia", "Rhodococcus", "Microbacterium", "Corynebacterium",
    "Bifidobacterium", "Actinomyces", "Frankia", "Propionibacterium",
    "Kitasatospora", "Leifsonia", "Cellulomonas", "Kineococcus",
    "Micromonospora", "Geodermatophilus", "Promicromonospora",
    "Actinoplanes", "Amycolatopsis", "Saccharomonospora",
    "Nocardioides", "Kocuria", "Actinobacteria",
}
resolved["taxon_class"] = resolved["genus"].map(
    lambda g: "Actinobacteria" if g in ACTINO else "Other"
)
n_actino  = (resolved["taxon_class"] == "Actinobacteria").sum()
n_other   = (resolved["taxon_class"] == "Other").sum()
a_mai     = ((resolved["taxon_class"] == "Actinobacteria") & (resolved["has_mai"] == 1)).sum()
o_mai     = ((resolved["taxon_class"] == "Other") & (resolved["has_mai"] == 1)).sum()

print(f"\nResolved taxonomy: {n_actino} Actinobacteria, {n_other} non-Actinobacteria (total {n_actino+n_other})")
print(f"MAI in Actinobacteria: {a_mai}/{n_actino} ({100*a_mai/n_actino:.1f}%)")
print(f"MAI in non-Actinobacteria: {o_mai}/{n_other} ({100*o_mai/n_other:.1f}%)")

ct = [[a_mai, n_actino - a_mai], [o_mai, n_other - o_mai]]
OR, pval = fisher_exact(ct)
print(f"\nFisher\'s exact (Actinobacteria vs. non-Actinobacteria):")
print(f"  Contingency: [[{a_mai}, {n_actino-a_mai}], [{o_mai}, {n_other-o_mai}]]")
print(f"  Odds ratio: {OR:.1f}")
print(f"  p-value: {pval:.2e}")

print("\nGenus breakdown for MAI-positive resolved genomes:")
print(resolved[resolved["has_mai"]==1].groupby(["taxon_class","genus"]).size()
      .sort_values(ascending=False).to_string())


Total genome records: 2,925
Unique organisms (de-duplicated on organism_name): 2,075
MAI-positive genome records: 163
MAI-positive unique strains: 136

Complete 4-gene pathway records: 134
Complete 4-gene pathway unique strains: 112

Complete-pathway genus breakdown:
  Environmental (unresolved): 51 unique strains
  Streptomyces          30 unique strains
  Arthrobacter           7 unique strains
  Rhodococcus            7 unique strains
  Actinobacteria         4 unique strains
  Nocardioides           4 unique strains
  Promicromonospora      3 unique strains
  Kitasatospora          1 unique strains
  Corynebacterium        1 unique strains
  Microbacterium         1 unique strains
  Leifsonia              1 unique strains
  Kocuria                1 unique strains
  Paenarthrobacter       1 unique strains

Resolved taxonomy: 76 Actinobacteria, 973 non-Actinobacteria (total 1049)
MAI in Actinobacteria: 62/76 (81.6%)
MAI in non-Actinobacteria: 24/973 (2.5%)

Fisher's exact (Actinobact

## Section 4 — Summary

In [8]:
print("\n" + "="*80)
print("ENIGMA Genome Depot Survey Complete")
print("="*80)
print(f"\nOutput files:")
print(f"  - {DATA_DIR / 'enigma_all_genomes.parquet'}")
print(f"  - {DATA_DIR / 'enigma_mai_hits.csv'}")
print(f"  - {DATA_DIR / 'enigma_pathway_completeness.csv'}")
print("\nReady for NB02: Candidate Ranking and Experimental Prioritisation")


ENIGMA Genome Depot Survey Complete

Output files:
  - /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/data/enigma_all_genomes.parquet
  - /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/data/enigma_mai_hits.csv
  - /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/data/enigma_pathway_completeness.csv

Ready for NB02: Candidate Ranking and Experimental Prioritisation
